In [15]:
import numpy as np
import torch.nn as nn
from tqdm import tqdm
from torch.distributions.multivariate_normal import MultivariateNormal
import torch
import matplotlib.pyplot as plt
from dotenv import load_dotenv; load_dotenv()
%cd {os.getenv('PROJECT_PATH')}

# from load import load_aggregate_datamed, load_datamed
# from input import get_params
# from Experiment import Experiment, prepare_experiments
# from Utils.MMD_utils import init_params
# #from Core.train_node import Trainer
# from Core.node import NeuralODE

/home/id94ebid/projects/continuous_GMM


In [ ]:
B = 16 # Batch size
T = 100 # Number of timesteps
F = 3 # Numbers of features
K = 2 # Number of mixture components
r = 128 # Estimate the covariance matrix
R = 7 # Number of eigenvalues
dt = 1/(T-1)
ts = torch.linspace(0, 1, T)
X = torch.randn((B, T)) #torch.randn((B, T, F))

sigma = 1.
alpha=-1/(2*sigma**2)

In [3]:
def rbf_fkernel(x1, x2, dt, alpha=-1.0): # (B, T, F)
    y = torch.square(x1-x2).sum(-1) # (B, T)
    norm = dt * (y[..., 0] / 2 + y[..., 1:-1].sum(dim=-1) + y[..., -1] / 2)
    return torch.exp(alpha*norm)

def rbf_kernel(x1, x2, sigma): # (B, F) or (B)
    norm = torch.square(x1-x2).sum(-1) # (B, T)
    return torch.exp(-1/(2*sigma**2)*norm)

In [4]:
m = torch.randn((K,T)) #torch.randn((K, F))
pi = torch.full((K,), 1/K)
kernels = [lambda x1, x2, s=sigma: rbf_kernel(x1, x2, sigma=s) for sigma in np.linspace(0.1,10, K)]

In [5]:
K_mat = torch.stack([kernel(ts[:, None, None], ts[None, :, None]) for kernel in kernels])

evals, evecs = torch.linalg.eigh(K_mat)
# eigh returns eigenvalues in ascending order, so take the last R
evals, evecs = evals[..., -R:], evecs[..., -R:]
evals, evecs = torch.flip(evals, dims=[-1]), torch.flip(evecs, dims=[-1])

In [6]:
def MMD(X, m, pi, evals, evecs, alpha):
    """
    X: [B, T]
    m: [K, T]
    evals: [K, R]
    evecs: [K, T, R]
    K_math: [K, T, T]
    """
    # E1
    E1 = torch.mean(rbf_fkernel(X[None, :, :, None], X[:, None, :, None], dt, alpha=alpha))

    # E2
    p1 = torch.prod(torch.sqrt(1 - 2*alpha*evals), dim=-1)
    a1 = X[:, None, :] - m[None, :, :]
    b1 = torch.sum(a1[..., None] * evecs[None], dim=2)*dt # TODO
    c1 = b1 / (1-2*alpha*evals[None])
    d1 = 1/B*torch.sum(torch.exp(alpha*torch.sum(c1, dim=-1)), dim=0)
    E2 = torch.sum(pi*(p1*d1))

    # E3
    p2 = torch.prod(torch.sqrt(1 - 2*alpha*(evals[None, :, :] + evals[:, None, :])), dim=-1)
    a2 = (m[None, :, :] - m[:, None, :])
    b2 = (evecs[None] + evecs[:, None])
    c2 = torch.sum(a2[..., None] * b2, dim=2)*dt # TODO
    d2 = c2 / (1-2*alpha*(evals[None]-evals[:, None]))
    e2 = torch.exp(torch.sum(alpha*d2, dim=-1))
    E3 = torch.sum(torch.outer(pi, pi)*p2*e2)
    
    return E1, E2, E3

In [13]:
m = nn.Parameter(torch.randn((K,T)))
pi = nn.Parameter(torch.full((K,), 1/K))
MMD(X, m, pi, evals, evecs, alpha=alpha)

(tensor(0.4218),
 tensor(1482.6647, grad_fn=<SumBackward0>),
 tensor(8110.5576, grad_fn=<SumBackward0>))

In [ ]:
optimizer = torch.optim.Adam([m, pi], lr=0.1)
pbar = tqdm(range(1000))
for i in pbar:
    optimizer.zero_grad()
    E1, E2, E3 = MMD(X, m, pi, evals, evecs, alpha=alpha)
    loss = E1 + E2 + E3
    loss.backward()
    optimizer.step()
    pi.data.clamp_(0.,1.)
    pi.data /= 1e-5+torch.sum(pi.data)
    pbar.set_description(f"E1: {E1.item():.4f}, E2: {E2.item():.4f}, E3: {E3.item():.4f}")

E1: 0.4218, E2: 2.3323, E3: 26.5500: 100%|██████████| 1000/1000 [00:03<00:00, 307.12it/s]


In [18]:
pi_cpu = pi.detach().numpy()
print(pi_cpu)

[0.         0.99998987 0.         0.         0.        ]


In [19]:
m

Parameter containing:
tensor([[ 2.3556e+00,  2.9385e+00,  2.5748e+00,  2.2710e+00,  2.7313e+00,
          9.0998e-01,  8.1442e-01,  2.2756e+00,  2.2042e+00,  1.2651e+00,
          4.9025e+00,  2.0723e+00,  5.4272e-01, -2.5576e+00, -3.3801e+00,
         -1.9008e+00, -1.6174e+00, -2.8050e+00,  2.4207e+00, -2.9737e+00,
          9.0686e-01, -2.8363e+00,  2.9845e+00, -1.2969e+00, -3.4772e+00,
          4.2925e+00, -8.7658e-01,  4.6677e+00,  4.0711e+00,  1.1685e+00,
          2.0365e+00,  1.4872e+00,  2.9757e+00,  1.3913e+00,  2.4463e+00,
          1.6468e+00, -2.6310e+00,  7.4062e-01, -3.3775e+00, -1.6345e+00,
          3.6661e+00,  2.1086e+00,  3.7502e+00, -5.1295e-01, -2.4208e+00,
         -6.2245e-01, -1.8654e+00, -1.1174e+00, -2.7431e+00, -3.8522e+00,
         -2.1686e+00, -2.9860e+00, -2.9375e+00, -1.4321e+00,  1.5101e+00,
         -6.4448e-01,  7.8621e-01,  3.0285e+00,  2.3453e+00,  3.3730e+00,
          1.7554e+00,  2.6257e+00,  1.9225e+00,  2.2852e+00,  2.7855e+00,
          3.2316